In [1]:
import sqlite3

conn = sqlite3.connect('../data/Chinook_Sqlite.sqlite')
cursor = conn.cursor()
print('Connected to Chinook database successfully!')

Connected to Chinook database successfully!


In [2]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [row[0] for row in cursor.fetchall()]

schema = {}
for table in tables:
    cursor.execute(f'PRAGMA table_info({table})')
    columns = cursor.fetchall()
    schema[table] = [(col[1], col[2]) for col in columns]

def format_schema_for_prompt(schema):
    schema_str = ''
    for table, columns in schema.items():
        schema_str += f'Table: {table}\n'
        for col_name, col_type in columns:
            schema_str += f'  - {col_name} ({col_type})\n'
        schema_str += '\n'
    return schema_str

schema_string = format_schema_for_prompt(schema)
print('Schema loaded successfully!')
print(f'Total tables: {len(schema)}')

Schema loaded successfully!
Total tables: 11


In [3]:
def build_prompt(schema_string, user_question):
    prompt = f"""You are an expert SQL assistant.
You are given the following database schema:
{schema_string}
Write a valid SQLite SQL query to answer the following question:
{user_question}
Rules:
- Return ONLY the SQL query, nothing else
- Do not include any explanation or markdown
- Do not use ```sql or ``` in your response
- Make sure the query is valid SQLite syntax
"""
    return prompt

# Note: this generic prompt (same format used for GPT-4o-mini and Qwen) 
# produced no output with SQLCoder — see below for the model-specific format that worked.
print('build_prompt function defined successfully!')

build_prompt function defined successfully!


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "defog/sqlcoder-7b-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("SQLCoder loaded successfully!")

/home/amarins.r/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/amarins.r/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  6.79it/s]
/home/amarins.r/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SQLCoder loaded successfully!


In [5]:
test_question = "Which artist has the most albums?"

In [6]:
sqlcoder_prompt = f"""### Task
Generate a SQL query to answer the following question:
`{test_question}`

### Database Schema
The query will run on a database with the following schema:
{schema_string}

### Answer
Given the database schema, here is the SQL query that answers `{test_question}`:
```sql
"""

inputs = tokenizer(sqlcoder_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
sqlcoder_sql = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(sqlcoder_sql)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT a."Name", COUNT(aa.AlbumId) AS album_count FROM Album aa JOIN Artist a ON aa.ArtistId = a.ArtistId GROUP BY a."Name" ORDER BY album_count DESC LIMIT 1;


In [7]:
def run_sql(sql_query):
    cursor.execute(sql_query)
    results = cursor.fetchall()
    return results

In [8]:
sqlcoder_results = run_sql(sqlcoder_sql)
print(sqlcoder_results)

[('Iron Maiden', 21)]


In [9]:
test_question_2 = "What are the top 5 genres by number of tracks?"

sqlcoder_prompt_2 = f"""### Task
Generate a SQL query to answer the following question:
{test_question_2}

### Database Schema
The query will run on a database with the following schema:
{schema_string}

### Answer
Given the database schema, here is the SQL query that answers the question:
"""

inputs = tokenizer(sqlcoder_prompt_2, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
sqlcoder_sql_2 = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(sqlcoder_sql_2)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT g.Name, COUNT(t.TrackId) AS TrackCount FROM Track t JOIN Genre g ON t.GenreId = g.GenreId GROUP BY g.Name ORDER BY TrackCount DESC LIMIT 5;


In [10]:
sqlcoder_results_2 = run_sql(sqlcoder_sql_2)
print(sqlcoder_results_2)

[('Rock', 1297), ('Latin', 579), ('Metal', 374), ('Alternative & Punk', 332), ('Jazz', 130)]


In [11]:
test_question_3 = "How many customers are from the USA?"

sqlcoder_prompt_3 = "### Task\nGenerate a SQL query to answer the following question:\n" + test_question_3 + "\n\n### Database Schema\nThe query will run on a database with the following schema:\n" + schema_string + "\n### Answer\nGiven the database schema, here is the SQL query that answers the question:\n"

inputs = tokenizer(sqlcoder_prompt_3, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

sqlcoder_sql_3 = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(sqlcoder_sql_3)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT COUNT(*) FROM Customer c WHERE c.Country ILIKE '%USA%';


In [12]:
try:
    sqlcoder_results_3 = run_sql(sqlcoder_sql_3)
    print(sqlcoder_results_3)
except Exception as e:
    print(f"SQLCoder query failed: {e}")

SQLCoder query failed: near "ILIKE": syntax error


In [13]:
error_message = "near \"ILIKE\": syntax error"

correction_prompt = "### Task\nThe following SQL query failed on SQLite:\n" + sqlcoder_sql_3 + "\n\nError message:\n" + error_message + "\n\nPlease fix the query. Only use valid SQLite syntax (no ILIKE, use LIKE or = instead).\n\n### Database Schema\n" + schema_string + "\n### Answer\nHere is the corrected SQLite query:\n"

inputs = tokenizer(correction_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
sqlcoder_sql_3_corrected = tokenizer.decode(new_tokens, skip_special_tokens=True)
print(sqlcoder_sql_3_corrected)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SELECT COUNT(*) FROM Customer c WHERE c.Country LIKE '%USA%';


In [14]:
sqlcoder_results_3_corrected = run_sql(sqlcoder_sql_3_corrected)
print(sqlcoder_results_3_corrected)

[(13,)]
